# 12 – Stats

Esplorazione e data cleaning del dataset `stats.csv`.

| Colonna | Descrizione |
|---|---|
| `mal_id` | ID dell'anime su MAL |
| `watching` | Numero di utenti che stanno guardando l'anime |
| `completed` | Numero di utenti che hanno completato l'anime |
| `on_hold` | Numero di utenti che hanno messo l'anime in pausa |
| `dropped` | Numero di utenti che hanno abbandonato l'anime |
| `plan_to_watch` | Numero di utenti che intendono guardare l'anime |
| `total` | Totale utenti che hanno l'anime in lista |
| `score_N_votes` | Numero di voti ricevuti per il punteggio N (1–10) |
| `score_N_percentage` | Percentuale di voti per il punteggio N rispetto al totale dei voti |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_stats = pd.read_csv('../datasets/stats.csv')
print(f'Shape: {df_stats.shape}')
df_stats.info()
df_stats.head()

**Osservazioni iniziali:**
- Il dataset contiene **28.955 righe** e **27 colonne**.
- Le prime 7 colonne (`mal_id` + conteggi utenti) sono complete senza null.
- Le 20 colonne `score_N_*` presentano 430 null, corrispondenti ad anime senza voti registrati su MAL (null strutturali).
- I tipi di dati saranno verificati nell'analisi per colonna.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_stats)

mask_dup = df_stats.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_stats[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_stats.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_stats):,}')

Nessun duplicato esatto trovato. Tutte le 28.955 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `mal_id`

Questa colonna è la **chiave primaria** del dataset (ogni riga corrisponde a un anime distinto) ed è anche una **chiave esterna** che referenzia `mal_id` di `details.csv`.

I controlli rilevanti sono:
- **Valori nulli**: non ammessi su una chiave primaria.
- **Duplicati**: non ammessi su una chiave primaria.
- **Integrità referenziale**: ogni ID deve esistere in `details_clean.csv`.

Usiamo `check_fk` per verificare i controlli d'integrità referenziale.

In [ ]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_mal = check_fk(df_stats['mal_id'], df_details['mal_id'], child_df=df_stats)

print(f'Null in mal_id      : {df_stats["mal_id"].isna().sum()}')
print(f'Duplicati in mal_id : {df_stats["mal_id"].duplicated().sum()}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Nessun duplicato**: la colonna è già una chiave primaria univoca.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia necessaria.**

### 2.2 `watching`

Numero di utenti che stanno attualmente guardando l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [ ]:
analyze(df_stats['watching'])

**Osservazioni:**
- Nessun valore null. Il dtype è già `int64` che è adeguato.
- I valori sono ≥ 0 come atteso per un conteggio. Stampiamo i valori estremi per verificare se si tratta di un'anomalia.

In [ ]:
df_details_url = pd.read_csv('../datasets/details.csv', usecols=['mal_id', 'title', 'url'])
estremi_watching = (
    df_stats[df_stats["watching"] >= 275702][["mal_id", "watching", "completed", "total"]]
    .sort_values("watching", ascending=False)
    .merge(df_details_url, on="mal_id", how="left")
)
print(f"Righe con watching >= 275,702: {len(estremi_watching)}")
estremi_watching

I valori estremi corrispondono ad anime molto famosi e qundi non si tratta di anomalie.

**Nessuna pulizia necessaria.**

### 2.3 `completed`

Numero di utenti che hanno completato l'anime.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [ ]:
analyze(df_stats['completed'])

**Osservazioni:**
- Nessun null. Il dtype già `int64` che è adeguato.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**

### 2.4 `on_hold`

Numero di utenti che hanno messo l'anime in pausa.

Colonna numerica intera. I duplicati sono **attesi**: più anime possono avere lo stesso conteggio. Usiamo `analyze`.

In [ ]:
analyze(df_stats['on_hold'])

**Osservazioni:**
- Nessun null. Dtype già `int64`.
- I valori sono ≥ 0 come atteso per un conteggio.

**Nessuna pulizia necessaria.**